In [15]:
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import lfilter

# -----------------------
# STABLE "LSF-LIKE" SPACE
# (FFT-based surrogate)
# -----------------------

def lpc_to_spectral_shape(a, n=256):
    # frequency response magnitude
    w, h = np.linspace(0, np.pi, n), None
    h = np.abs(np.polyval(a[::-1], np.exp(-1j * w)))
    h = np.log(h + 1e-8)
    return h


def spectral_to_control_points(spec, order):
    # downsample spectrum into stable control points
    idx = np.linspace(0, len(spec) - 1, order)
    return np.interp(idx, np.arange(len(spec)), spec)


def control_to_lpc(ctrl):
    # smooth reconstruction of LPC-like envelope
    # (not exact LPC inverse, but stable all-pole approximation)
    a = np.polyfit(np.arange(len(ctrl)), ctrl, len(ctrl) - 1)
    a = np.real(a)

    # stabilize
    a = np.insert(a, 0, 1.0)
    return a / (np.abs(a[0]) + 1e-8)

# -----------------------
# load audio
# -----------------------

y, sr = librosa.load("../raw_audio/dummy.wav", sr=None)

frame_length = int(0.03 * sr)
hop_length = frame_length // 2
order = 16

y = np.pad(y, (0, frame_length), mode="constant")

frames = librosa.util.frame(y, frame_length=frame_length, hop_length=hop_length)

window = np.hanning(frame_length)

output = np.zeros(len(y))
norm = np.zeros(len(y))

prev_ctrl = None

# -----------------------
# smooth control params
# -----------------------

smooth_rate = 0.1
drift_strength = 0.05
noise_strength = 0.01

# -----------------------
# main loop
# -----------------------

for i in range(frames.shape[1]):

    frame = frames[:, i] * window

    # LPC analysis
    a = librosa.lpc(frame, order=order)

    # -----------------------
    # STABLE SPECTRAL CONTROL
    # -----------------------

    spec = lpc_to_spectral_shape(a)
    ctrl = spectral_to_control_points(spec, order + 1)

    # temporal smoothing
    if prev_ctrl is not None:
        ctrl = (1 - smooth_rate) * ctrl + smooth_rate * prev_ctrl

    # smooth drift (low-frequency motion)
    drift = np.sin(np.linspace(0, np.pi, len(ctrl)) + i * 0.05)
    ctrl = ctrl + drift_strength * drift

    # correlated noise (safe)
    noise = np.random.randn(len(ctrl))
    noise = np.convolve(noise, [1, 2, 1], mode="same") / 4
    ctrl = ctrl + noise_strength * noise

    prev_ctrl = ctrl

    # -----------------------
    # stable LPC reconstruction
    # -----------------------

    a = control_to_lpc(ctrl)

    # synthesis
    residual = lfilter(a, [1.0], frame)
    reconstructed = lfilter([1.0], a, residual)

    start = i * hop_length

    output[start:start + frame_length] += reconstructed * window
    norm[start:start + frame_length] += window**2

output /= (norm + 1e-8)

sf.write("robust_experimental_lpc.wav", output, sr)

/var/folders/9m/wg7rj6y54778yxmyj6wjc25c0000gn/T/ipykernel_27901/2423509654.py:112: RuntimeWarning: overflow encountered in divide
  output /= (norm + 1e-8)
